In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import missingno as msno
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import ttest_rel
from sklearn.compose import TransformedTargetRegressor

In [3]:
df = pd.read_csv('Data.csv')

In [4]:
X = df.drop(['Dev Time (Days)', 'issue_key','Current Status'], axis=1)
y = np.log1p(df['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.to_string(max_rows=None))

Model trained successfully!
Mean Absolute Error (MAE): 0.00
R-squared Score: 1.00
Unnamed: 0                                  9.999706e-01
Start Date                                  1.395040e-05
total_swag                                  3.221070e-06
impacted_area_(none)                        1.035479e-06
subtask_count                               1.031424e-06
fix_version_ASTRO_SR2024.3                  8.120946e-07
fix_version_(no version)                    7.906103e-07
technology_Aloha                            6.600343e-07
technology_APX                              5.068946e-07
technology_(none)                           5.014239e-07
affects_version_(no version)                4.768878e-07
impacted_area_Box Test                      4.583176e-07
fix_version_No_SW_Release                   4.323605e-07
impacted_area_CPE-EE                        4.172215e-07
technology_APX Next                         3.633345e-07
affects_version_ASTRO_SR2021.1              3.135887e-07
Applic

# Test Multiple Imputation Methods for total_swag (After Isolation Forest)

In [5]:
df = df.copy()
num_cols = df.select_dtypes(include='number').columns
assert 'total_swag' in num_cols, "total_swag must be numeric."
df_num = df[num_cols]

In [6]:
obs_idx = df_num['total_swag'].dropna().index
rng = np.random.default_rng(42)
val_size = max(1, int(0.2 * len(obs_idx)))
val_idx = rng.choice(obs_idx, size=val_size, replace=False)
df_test = df_num.copy()
y_true = df_test.loc[val_idx, 'total_swag'].to_numpy()
df_test.loc[val_idx, 'total_swag'] = np.nan

Bayesian Ridge (features scaled)

In [7]:
est = make_pipeline(StandardScaler(), BayesianRidge())
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=est, sample_posterior = True, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 8603.817 RMSE: 164615352.541 Mean imputation SD: 21131.820


Linear Regression

In [8]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=LinearRegression(), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 12208.212 RMSE: 540011482.962 Mean imputation SD: 0.000


Decision Tree

In [9]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=DecisionTreeRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 6992.314 RMSE: 522901378.342 Mean imputation SD: 1691.917


Random Forest

In [10]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5099.257 RMSE: 239918015.810 Mean imputation SD: 474.499


Extra Trees

In [11]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5158.008 RMSE: 206552555.439 Mean imputation SD: 444.331


Gradient boosting

In [12]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=GradientBoostingRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5871.561 RMSE: 307733960.735 Mean imputation SD: 287.564


K Neighbours

In [13]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=KNeighborsRegressor(n_neighbors=5), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5579.506 RMSE: 157385110.283 Mean imputation SD: 0.000


Choose Random Forest

In [14]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

m = 10  
imputed_datasets = []
use = df.drop(['issue_key', 'Current Status'], axis=1)

print(f"--- Starting Multiple Imputation (m={m}) ---")

for i in range(m):
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1),
        max_iter=10,
        random_state=i,
        sample_posterior=False 
    )
    
    imputed_array = imputer.fit_transform(use)
    
    df_temp = pd.DataFrame(imputed_array, columns=use.columns, index=use.index)
    imputed_datasets.append(df_temp)
    
    print(f"Imputation {i+1}/{m} complete.")


df_final = pd.concat(imputed_datasets).groupby(level=0).mean()

--- Starting Multiple Imputation (m=10) ---
Imputation 1/10 complete.
Imputation 2/10 complete.
Imputation 3/10 complete.
Imputation 4/10 complete.
Imputation 5/10 complete.
Imputation 6/10 complete.
Imputation 7/10 complete.
Imputation 8/10 complete.
Imputation 9/10 complete.
Imputation 10/10 complete.


In [15]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.impute import KNNImputer

df_original = df
observed_values = df_original['total_swag'].dropna()

imputed_full_dataset = df_final['total_swag']


statistic, p_value = ks_2samp(observed_values, imputed_full_dataset)

print(f"KS Statistic for Feature_X: {statistic:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05

if p_value > alpha:
    print("Interpretation: Fail to reject the null hypothesis.")
    print("The distributions are statistically similar. The imputation is VALID.")
else:
    print("Interpretation: Reject the null hypothesis.")
    print("The distributions are significantly different. The imputation may have introduced BIAS.")

KS Statistic for Feature_X: 0.3882
P-value: 0.0000
Interpretation: Reject the null hypothesis.
The distributions are significantly different. The imputation may have introduced BIAS.


In [17]:
df_final.to_csv('imputed.csv',index=False)